This project scope is to simulate monitoring framework *designed to assess market data reliability before risk metrics and management reporting are produced*.

It combines *field-level completeness checks*, *validation rules*, *stale/outlier detection*, and *basic risk alerts* into an operational monitoring layer.

In [5]:
import pandas as pd
import numpy as np
import yfinance as yf

start_date = "2022-01-01"
tickers = ["SPY", "AAPL", "TLT", "GLD", "EURUSD=X"]

raw = yf.download(
    tickers=tickers,
    start=start_date,
    auto_adjust=False,
    progress=False,
    group_by="ticker",
    threads=True
)

frames = []

for ticker in tickers:
    if ticker not in raw.columns.get_level_values(0):
        print(f"{ticker} not found in download")
        continue

    df_t = raw[ticker].copy().reset_index()
    df_t.columns = [str(c).lower().replace(" ", "_") for c in df_t.columns]

    # standardize adj close name
    if "adj_close" not in df_t.columns and "adj close" in df_t.columns:
        df_t = df_t.rename(columns={"adj close": "adj_close"})

    df_t["asset"] = ticker

    cols = ["date", "asset", "open", "high", "low", "close", "adj_close", "volume"]
    cols_existing = [c for c in cols if c in df_t.columns]

    df_t = df_t[cols_existing]
    frames.append(df_t)

market_data = pd.concat(frames, ignore_index=True)
market_data["date"] = pd.to_datetime(market_data["date"])
market_data = market_data.sort_values(["asset", "date"]).reset_index(drop=True)

In [8]:
from src_functions import cleaning

market_clean = cleaning(market_data)
market_clean

,date,asset,open,high,low,close,adj_close,volume,asset_class,ccy,fx_rate,spread,bid,ask,source,week_day,business_day
0,2022-01-03,AAPL,177.830002,182.880005,177.710007,182.009995,178.103638,104487900.0,Equity,USD,1.0,0.27301,181.873490,182.146500,Bloomberg,Monday,True
1,2022-01-04,AAPL,182.630005,182.940002,179.119995,179.699997,175.843216,99310400.0,Equity,USD,1.0,0.26955,179.565222,179.834772,Bloomberg,Tuesday,True
2,2022-01-05,AAPL,179.610001,180.169998,174.639999,174.919998,171.165787,94537600.0,Equity,USD,1.0,0.26238,174.788808,175.051188,Bloomberg,Wednesday,True
3,2022-01-06,AAPL,172.699997,175.300003,171.639999,172.000000,168.308533,96904000.0,Equity,USD,1.0,0.25800,171.871000,172.129000,Bloomberg,Thursday,True
4,2022-01-07,AAPL,172.889999,174.139999,171.029999,172.169998,168.474869,86709100.0,Equity,USD,1.0,0.25825,172.040873,172.299123,Bloomberg,Friday,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5500,2026-03-23,TLT,86.180000,86.720001,85.940002,86.389999,86.389999,69686700.0,Fixed Income,USD,1.0,0.06911,86.355444,86.424554,Reuters,Monday,True
5501,2026-03-24,TLT,85.720001,86.290001,85.559998,86.010002,86.010002,51799400.0,Fixed Income,USD,1.0,0.06881,85.975597,86.044407,Reuters,Tuesday,True
5502,2026-03-25,TLT,86.750000,86.879997,86.480003,86.839996,86.839996,37861000.0,Fixed Income,USD,1.0,0.06947,86.805261,86.874731,Reuters,Wednesday,True
5503,2026-03-26,TLT,86.339996,86.610001,85.930000,86.110001,86.110001,39537000.0,Fixed Income,USD,1.0,0.06889,86.075556,86.144446,Reuters,Thursday,True


### Key Assumption

*Bid* and *ask* represent the *best available prices* in the market at which an instrument *can be sold or bought* immediately.

Since Yahoo Finance only provides end-of-day prices, bid-ask fields were synthetically created around the close price to simulate a more realistic monitoring environment.

*Spread assumptions were defined using a simple liquidity-based logic*: 

    - more liquid instruments are assigned tighter spreads

    - less liquid ones receive slightly wider spreads.

The purpose is not to replicate exact market conditions, but to create a coherent structure for data quality analysis.

*Short note on sources*

Data sources were assigned synthetically to simulate a multi-provider environment.
This makes the dataset more suitable for governance-style controls, such as checking whether certain issues are concentrated in specific providers.


In [9]:
from src_functions import inject_data_quality_issues

market_data_dirty = inject_data_quality_issues(market_clean)
market_data_dirty

,date,asset,open,high,low,close,adj_close,volume,asset_class,ccy,fx_rate,spread,bid,ask,source,week_day,business_day
0,2022-01-03,AAPL,177.830002,182.880005,177.710007,182.009995,178.103638,NaN,Equity,USD,1.0,0.27301,181.873490,182.146500,Bloomberg,Monday,True
1,2022-01-04,AAPL,182.630005,182.940002,179.119995,179.699997,175.843216,NaN,Equity,USD,1.0,NaN,NaN,179.834772,Bloomberg,Tuesday,True
2,2022-01-05,AAPL,179.610001,180.169998,174.639999,174.919998,171.165787,NaN,Equity,USD,1.0,0.26238,174.788808,175.051188,NaN,Wednesday,True
3,2022-01-06,AAPL,172.699997,175.300003,171.639999,172.000000,168.308533,NaN,Equity,USD,1.0,NaN,NaN,172.129000,NaN,Thursday,True
4,2022-01-07,AAPL,172.889999,174.139999,171.029999,172.169998,168.474869,NaN,Equity,USD,1.0,0.25825,172.040873,172.299123,Bloomberg,Friday,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5500,2026-03-23,TLT,86.180000,86.720001,85.940002,86.389999,86.389999,69686700.0,Fixed Income,USD,1.0,0.06911,86.355444,86.424554,Reuters,Monday,True
5501,2026-03-24,TLT,85.720001,86.290001,85.559998,86.010002,86.010002,51799400.0,Fixed Income,USD,1.0,NaN,85.975597,NaN,Reuters,Tuesday,True
5502,2026-03-25,TLT,86.750000,86.879997,86.480003,86.839996,86.839996,37861000.0,Fixed Income,USD,1.0,NaN,86.805261,NaN,Reuters,Wednesday,True
5503,2026-03-26,TLT,86.339996,86.610001,85.930000,86.110001,86.110001,39537000.0,Fixed Income,USD,1.0,NaN,86.075556,NaN,NaN,Thursday,True


A controlled set of data quality issues (missing values, outliers, stale prices and bid/ask inconsistencies) was introduced to simulate a realistic market data environment and enable testing of data quality controls.